# Week 3 — Fundamentals of Data Engineering

**Wide World Importers | DuckDB OLTP → OLAP**

This notebook documents the successful Google Colab execution of the Week 3 laboratory. The SQL transformations and orchestration code are maintained in the [Week 3 GitHub branch](https://github.com/callMeJZ/analytics-solutions-bootcamp/tree/feat/week3-laboratory). Outputs below are preserved from the original executed notebook.

**Prerequisite:** Place the bootcamp repository folder in Google Drive at `MyDrive/analytics-solutions-bootcamp-main`. Running from a fresh runtime requires the repository folder and dataset download. The recorded run reused an existing OLTP database.

## 1. Mount Google Drive and set the project directory

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")
project_dir = Path("/content/drive/MyDrive/analytics-solutions-bootcamp-main/03_case_study/03_cs_data_engineering_guide")
assert project_dir.is_dir(), "Project directory missing in Google Drive"
os.chdir(project_dir)
print("Working directory:", Path.cwd())

## 2. Install dependencies

Install the dependencies defined by the Week 3 project.

In [ ]:
%pip install -r requirements.txt

## 3. Download the source CSVs

The original run downloaded the Wide World Importers CSVs to the project `data/` directory. Skip this step when the dataset is already present.

In [ ]:
!python src/00_download_csv.py

Current working directory: /content/drive/MyDrive/analytics-solutions-bootcamp-main/03_case_study/03_cs_data_engineering_guide
100% 15.8M/15.8M [00:00<00:00, 177MB/s]
Extracting files...
Successfully downloaded Kaggle data files to data.


## 4. Verify source dataset

In [ ]:
csv_files = sorted((project_dir / "data").glob("*.csv"))
print("Total CSV files:", len(csv_files))
print("Application.Cities.csv exists:", (project_dir / "data" / "Application.Cities.csv").is_file())

## 5. Build OLTP database

The ingestion script loads the CSV data into `asb_oltp.duckdb`. This was executed successfully in the original notebook.

In [ ]:
!python src/01_ingest_csv.py

Working directory: /content/drive/MyDrive/analytics-solutions-bootcamp-main/03_case_study/03_cs_data_engineering_guide
Starting OLTP ingestion...
Connected to DuckDB database: asb_oltp.duckdb
Creating schemas for Wide World Importers dataset...
100% 5/5 [00:00<00:00, 335.71it/s]
Done creating schemas for Wide World Importers dataset...
Creating tables for Wide World Importers dataset...
100% 29/29 [00:09<00:00,  2.91it/s]
Done creating tables for World Wide Importers OLTP.


## 6. Verify OLTP tables

In [1]:
import duckdb
from pathlib import Path

# Locate the saved database in Google Drive
db_path = Path(
    "/content/drive/MyDrive/analytics-solutions-bootcamp-main/"
    "03_case_study/03_cs_data_engineering_guide/"
    "asb_oltp.duckdb"
)

# Open the existing database in read-only mode
conn = duckdb.connect(str(db_path), read_only=True)

# Count the imported tables
result = conn.execute("""
    SELECT COUNT(*)
    FROM information_schema.tables
    WHERE table_schema NOT IN (
        'information_schema',
        'pg_catalog'
    )
    AND table_type = 'BASE TABLE'
""").fetchone()

print("OLTP DATABASE TABLE VERIFICATION")
print("-" * 40)
print("Total tables found:", result[0])

conn.close()

OLTP DATABASE TABLE VERIFICATION
----------------------------------------
Total tables found: 29


## 7. Synchronize Week 3 SQL and orchestration from GitHub

This records the exact implementation branch used in the successful run. When submitting from `main` after merging, update the branch name here if desired.

In [2]:
from pathlib import Path
import requests

project = Path(
    "/content/drive/MyDrive/analytics-solutions-bootcamp-main/"
    "03_case_study/03_cs_data_engineering_guide"
)

base_url = (
    "https://raw.githubusercontent.com/callMeJZ/"
    "analytics-solutions-bootcamp/feat/week3-laboratory/"
    "03_case_study/03_cs_data_engineering_guide/"
)

files = [
    "src/sql/olap/dim_city.sql",
    "src/sql/olap/dim_customer.sql",
    "src/sql/olap/dim_employee.sql",
    "src/sql/olap/dim_stock_item.sql",
    "src/sql/olap/fact_order.sql",
    "src/sql/olap/fact_sale.sql",
    "run_pipe.py"
]

assert (project / "asb_oltp.duckdb").is_file(), "OLTP database not found!"

# Download everything before replacing existing files.
updates = {}
for file in files:
    response = requests.get(base_url + file, timeout=30)
    response.raise_for_status()
    updates[file] = response.content

for file, content in updates.items():
    destination = project / file
    destination.write_bytes(content)
    print("Updated:", file)

print("\nAll 7 files synchronized successfully!")

Updated: src/sql/olap/dim_city.sql
Updated: src/sql/olap/dim_customer.sql
Updated: src/sql/olap/dim_employee.sql
Updated: src/sql/olap/dim_stock_item.sql
Updated: src/sql/olap/fact_order.sql
Updated: src/sql/olap/fact_sale.sql
Updated: run_pipe.py

All 7 files synchronized successfully!


## 8. Execute OLAP pipeline

The `--olap-only` option reuses the completed OLTP database and creates/refreshes `asb_olap.duckdb`.

In [3]:
!python run_pipe.py --olap-only

/content/drive/MyDrive/analytics-solutions-bootcamp-main/03_case_study/03_cs_data_engineering_guide

Running 02_ingest_olap.py...
Creating tables for Wide World Importers OLAP tables...
100% 7/7 [00:01<00:00,  5.83it/s]

Pipeline completed successfully.


## 9. Validate dimension and fact tables

All six required dimension/fact tables were populated. These row counts verify successful loading, not full data-quality or referential-integrity checks.

In [4]:
import duckdb

conn = duckdb.connect("asb_olap.duckdb", read_only=True)

tables = [
    "dim_city",
    "dim_customer",
    "dim_employee",
    "dim_stock_item",
    "fact_order",
    "fact_sale"
]

for table in tables:
    count = conn.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]
    print(f"{table}: {count:,} rows")

conn.close()

dim_city: 37,940 rows
dim_customer: 663 rows
dim_employee: 19 rows
dim_stock_item: 227 rows
fact_order: 231,412 rows
fact_sale: 228,265 rows


## Results

| Table | Observed rows |
| --- | ---: |
| `dim_city` | 37,940 |
| `dim_customer` | 663 |
| `dim_employee` | 19 |
| `dim_stock_item` | 227 |
| `fact_order` | 231,412 |
| `fact_sale` | 228,265 |

**Outcome:** `Pipeline completed successfully.` The OLTP database contained 29 imported tables. Database and CSV files remain in Google Drive; they are not embedded in this notebook.